Install Transformers and dataset library for processing. Once you have installed and need to rerun for the same session, you can comment it out. Use a "#" in front of the line to comment out.

In [55]:
!pip install datasets transformers

If you’re running this notebook on your own machine, double-check that your environment has the **latest versions** of the required libraries installed.

To **share your model on Hugging Face** and enable others to use it (including generating results through the **Hugging Face Inference API**), you’ll need to complete a couple of extra setup steps:

1. **Create a Hugging Face account** (if you don’t already have one).
2. **Get your authentication token** from your Hugging Face account settings.
3. You will get it in settings> Access Tokens.
4. Run the next cell in the notebook and **log in when prompted** with the token.

Once you’re authenticated, the notebook can upload your model and connect it to the Inference API.


In [56]:
from huggingface_hub import notebook_login

notebook_login()

Then you need to install Git-LFS. Git LFS (Large File Storage) is a Git extension that’s designed for tracking large files (like datasets, images, videos, audio, model weights, .pt/.bin/.ckpt, .zip, etc.) without bloating your Git repository.


In [57]:
!apt install git-lfs

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
git-lfs is already the newest version (3.0.2-1ubuntu0.3).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


Make sure your version of Transformers is at least 4.11.0 since the functionality was introduced in that version:

In [58]:
import transformers

print(transformers.__version__)

5.0.0


# Fine-tuning a language model

In this notebook, you’ll learn how to **fine-tune a Transformers model** for a **language modeling** task. We’ll focus on two common types of language modeling: Causal Language Modeling (CLM) and Masked language modeling (MLM)

**1) Causal Language Modeling (CLM)**
In causal language modeling, the model learns to **predict the next token** in a sequence.

* The input is a sentence like:
  *“The cat sat on the …”*
  and the model tries to predict what comes next.

* During training, the **labels are basically the same as the input**, just **shifted by one position** (so the model is always predicting the “next” token).

* To make sure the model doesn’t “peek” at future words, we use an **attention mask**.
  This mask forces the model to only look at the tokens **up to the current position**.
  So when predicting token *i+1*, it can only use tokens **1 through i**, not anything after that.


In [59]:
from IPython.display import Image, display
display(Image(
    url="https://raw.githubusercontent.com/nuzaeromar/LLM-demo/main/cat_full_generation.png",
    width=600,
    height=350
))


**Masked Language Modeling (MLM)** works a bit differently from causal language modeling. Instead of predicting the *next* word, the model learns to **fill in missing words**.

* Some tokens in the input sentence are **masked** (for example, replaced with `[MASK]`).
* The model’s job is to **predict the original tokens** that were hidden.
* Unlike causal models, the model can see the **entire sentence**—both the words **before and after** the masked token—so it can use full context to make a better prediction.



In [60]:
from IPython.display import Image, display
display(Image(
    url="https://raw.githubusercontent.com/nuzaeromar/LLM-demo/main/cat_mlm.png",
    width=400,
    height=500
))

In this notebook, we’ll walk through how to:

* **Load and preprocess datasets** for both language modeling tasks.
* **Prepare the inputs and labels** in the right format.
* **Fine-tune a model using the `Trainer` API**, which handles training, evaluation, and logging with minimal boilerplate code.

By the end, you’ll have a clear, hands-on understanding of how to train language models for different objectives using Hugging Face tools.

## Preparing the dataset

For each of those tasks, we will use the Wikitext2 dataset as an example. You can load it very easily with the Datasets library.


In [61]:
from datasets import load_dataset
datasets = load_dataset('wikitext', 'wikitext-2-raw-v1')

We check the number of samples available for training, validation and testing.

In [62]:
for split in datasets:
    print(split, len(datasets[split]))


test 4358
train 36718
validation 3760


Since, we have limited time, we'll be experimenting on a smaller dataset.

In [63]:
from datasets import DatasetDict

small_datasets = DatasetDict({
    "train": datasets["train"].select(range(100)),
    "validation": datasets["validation"].select(range(50)),
    "test": datasets["test"].select(range(50)),
})

for split in small_datasets:
    print(split, len(small_datasets[split]))


train 100
validation 50
test 50


You’re not limited to the dataset used above. You can **easily switch to a different dataset**—either one that’s already hosted on the Hugging Face Hub or your **own local files**.

To do this:

* Simply **uncomment the next cell** in the notebook.
* Replace the example paths with the **actual paths to your dataset files** (local paths or Hub dataset identifiers).

Once updated, the notebook will load and use your chosen dataset automatically, with no other changes needed.


In [64]:
# datasets = load_dataset("text", data_files={"train": path_to_train.txt, "validation": path_to_validation.txt}

To access an actual element, you need to select a split first, then give an index. Let's visualize a data element.

In [65]:
small_datasets["train"][9]

{'text': " As with previous Valkyira Chronicles games , Valkyria Chronicles III is a tactical role @-@ playing game where players take control of a military unit and take part in missions against enemy forces . Stories are told through comic book @-@ like panels with animated character portraits , with characters speaking partially through voiced speech bubbles and partially through unvoiced text . The player progresses through a series of linear missions , gradually unlocked as maps that can be freely scanned through and replayed as they are unlocked . The route to each story location on the map varies depending on an individual player 's approach : when one option is selected , the other is sealed off to the player . Outside missions , the player characters rest in a camp , where units can be customized and character growth occurs . Alongside the main story missions are character @-@ specific sub missions relating to different squad members . After the game 's completion , additional

To get a sense of what the data looks like, the following function will show some examples picked randomly in the dataset.

In [66]:
from datasets import ClassLabel
import random
import pandas as pd
from IPython.display import display, HTML

def show_random_elements(dataset, num_examples=10):
    assert num_examples <= len(dataset), "Can't pick more elements than there are in the dataset."
    picks = []
    for _ in range(num_examples):
        pick = random.randint(0, len(dataset)-1)
        while pick in picks:
            pick = random.randint(0, len(dataset)-1)
        picks.append(pick)

    df = pd.DataFrame(dataset[picks])
    for column, typ in dataset.features.items():
        if isinstance(typ, ClassLabel):
            df[column] = df[column].transform(lambda i: typ.names[i])
    display(HTML(df.to_html()))

In [67]:
show_random_elements(small_datasets["train"])

,text
0,Hall 's carbines 267 \n
1,
2,"Senjō no Valkyria 3 : Unrecorded Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the "" Nameless "" , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit "" Calamaty Raven "" . \n"
3,"A "" Summary of the Work Done for November , 1862 , Little Rock Arsenal "" shows : Fabrication : \n"
4,"On its day of release in Japan , Valkyria Chronicles III topped both platform @-@ exclusive and multi @-@ platform sales charts . By early February , the game sold 102 @,@ 779 units , coming in second overall to The Last Story for the Wii . By the end of the year , the game had sold just over 152 @,@ 500 units . \n"
5,"Unlike its two predecessors , Valkyria Chronicles III was not released in the west . According to Sega , this was due to poor sales of Valkyria Chronicles II and the general unpopularity of the PSP in the west . An unofficial fan translation patch began development in February 2012 : players with a copy of Valkyria Chronicles III could download and apply the patch , which translated the game 's text into English . Compatible with the Extra Edition , the patch was released in January 2014 . \n"
6,= = = Release = = = \n
7,"Partly due to these events , and partly due to the major losses in manpower Gallia suffers towards the end of the war with the Empire , the Nameless are offered a formal position as a squad in the Gallian Army rather than serve as an anonymous shadow force . This is short @-@ lived , however , as following Maximilian 's defeat , Dahau and Calamity Raven move to activate an ancient Valkyrian super weapon within the Empire , kept secret by their benefactor . Without the support of Maximilian or the chance to prove themselves in the war with Gallia , it is Dahau 's last trump card in creating a new Darcsen nation . As an armed Gallian force invading the Empire just following the two nations ' cease @-@ fire would certainly wreck their newfound peace , Kurt decides to once again make his squad the Nameless , asking Crowe to list himself and all under his command as killed @-@ in @-@ action . Now owing allegiance to none other than themselves , the 422nd confronts Dahau and destroys the Valkyrian weapon . Each member then goes their separate ways in order to begin their lives anew . \n"
8,= = Plot = = \n
9,"Major General Thomas C. Hindman , sent to command the district of Arkansas in May , 1862 , found the state nearly destitute of military material . Hindman established another armory at Arkadelphia , and revived the Little Rock Arsenal as a collection point and depot for armaments and ammunition manufacture for small arms . Hindman recorded : \n"


We can see, some of the texts are a full paragraph of a Wikipedia article while others are just titles or empty lines.

## Causal Language modeling

For **causal language modeling (CLM)**, we’ll prepare the data in a way that helps the model learn how text naturally flows.

Here’s the idea, step by step:

* First, we **tokenize all the text** in the dataset.
* Then, we **concatenate everything together** into one long stream of tokens.
* Next, we **split this stream into fixed-length chunks** (for example, sequences of a certain number of tokens).

Each training example the model sees is just a **continuous chunk of text**, which might look like:

* a piece taken entirely from the middle of one document, or
* the end of one document followed by a **beginning-of-sequence token** and the start of the next document.

For training:

* The **labels are the same as the input tokens**, but **shifted one position to the left**, so the model always learns to predict the next token.

In this example, we’ll use the **`distilgpt2`** model because it’s lightweight and fast to train.
That said, you’re free to choose **any other compatible checkpoint** from the list provided if you want to experiment with a different model.


In [68]:
model_checkpoint = "distilgpt2"

To make sure our text is tokenized in a way the model understands, we need to use **the same vocabulary that was used when the model was originally trained**.

Hugging Face makes this easy with the **`AutoTokenizer`** class. It automatically downloads the correct pretrained tokenizer for the model you choose and handles all the tokenization details for you.

In short, `AutoTokenizer` ensures that:

* Your text is split into tokens **exactly the way the model expects**
* The token IDs match the model’s learned embeddings
* You don’t have to worry about tokenizer-specific rules or settings

With just a few lines of code, you’ll be ready to preprocess your data consistently and correctly.


In [69]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)

Now we’re ready to **apply the tokenizer to all of our text data**. This step is straightforward thanks to the **Datasets** library.

We’ll use the dataset’s **`map`** method, which lets us apply a function to every example efficiently.

The process looks like this:

* First, we **define a small function** that takes a batch of text and runs it through the tokenizer.
* Then, we use `map` to **apply this function to the entire dataset**, converting raw text into token IDs the model can work with.

This approach is clean, fast, and scales well, even for large datasets.


Next, we apply this tokenization function to **all splits of the dataset** (for example, train and validation).

To make preprocessing faster and more efficient:

* We set **`batched=True`** so the tokenizer processes multiple examples at once.
* We use **4 processes** to parallelize the work and speed things up.

After tokenization, the original **text column is no longer needed**, so we remove it to keep the dataset clean and lightweight.

At the end of this step, the dataset contains only the tokenized inputs, ready to be used for training.


In [70]:
def tokenize_function(examples):
    return tokenizer(examples["text"])

In [71]:
tokenized_datasets = small_datasets.map(
    tokenize_function,
    batched=True,
    num_proc=4,
    remove_columns=["text"]
)

If we now look at an element of our datasets, we will see the text have been replaced by the input_ids the model will need:

In [72]:
tokenized_datasets["train"][1]

{'input_ids': [796, 569, 18354, 7496, 17740, 6711, 796, 220, 198],
 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1]}

Now comes the trickier part, that is the preprocessing step for CLM: we want the model to train on **continuous text**, not separate rows.

So we’ll do two things:

1. **Concatenate** all tokenized text together into one long stream of tokens.
2. **Split** that long stream into fixed-length chunks of size `block_size` (for example, 128 tokens).

We’ll use `dataset.map(..., batched=True)` again because batching gives us a useful superpower:
when `batched=True`, our mapping function can **return a different number of examples than it receives**. That means we can take a batch of many short sequences, merge them, and then output fewer (or more) chunked sequences.

Next, we choose a chunk length:

* Ideally, we start from the model’s **maximum context length** (the max sequence length it was pretrained with).
* But that maximum can be too large for your GPU memory.
* So for this notebook, we’ll use a smaller, GPU-friendly value: **`block_size = 128`**.

This creates training samples that are manageable in size while still giving the model meaningful context to learn from.


In [73]:
# block_size = tokenizer.model_max_length
block_size = 128

Then we write the preprocessing function that will group our texts:

In [74]:
def group_texts(examples):
    # Concatenate all texts.
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    # We drop the small remainder, we could add padding if the model supported it instead of this drop, you can
    # customize this part to your needs.
    total_length = (total_length // block_size) * block_size
    # Split by chunks of max_len.
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    result["labels"] = result["input_ids"].copy()
    return result

A couple of important details to keep in mind here:

* **We copy the input tokens to create the labels.**
  That’s because Transformers’ causal language models handle the “shift” internally during training (they automatically align inputs and labels so the model predicts the next token). So we **don’t need to manually shift labels ourselves**.

* **`map()` processes data in batches (default is 1,000 tokens per batch).**
  Since we’re concatenating tokens and then splitting into fixed-size chunks (`block_size`), the total number of tokens in each batch might not divide evenly by `block_size`.

  To keep things clean, we **drop the leftover tokens** at the end of each batch so that the length is always a perfect multiple of `block_size`. This avoids creating a final chunk that’s too short.

Intuition with a concrete example

Let’s say:
Average tokens per sample ≈ 50

block_size = 128

Batch size = 100

Tokens per batch ≈ 5,000

5,000 % 128 = 8 leftover tokens → dropped

Batch size = 1,000

Tokens per batch ≈ 50,000

50,000 % 128 = 80 leftover tokens → dropped


* **You can control this behavior.** Adjust batch size according to wish.

* **You can also speed things up with multiprocessing.**
  Using multiple processes lets tokenization and chunking run in parallel, which can significantly reduce preprocessing time on larger datasets.


In [75]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

And we can check our datasets have changed: now the samples contain chunks of block_size contiguous tokens, potentially spanning over several of our original texts.

In [76]:
tokenizer.decode(lm_datasets["train"][1]["input_ids"])

' game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " Calamaty Raven " . \n The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving for series newcomers . Character designer Raita Honjou and composer Hitoshi Sakimoto both returned from previous entries , along with Valkyria Chronicles II director Takeshi Oz'

Now that the data has been cleaned, we're ready to instantiate our `Trainer` with a model:

In [77]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(model_checkpoint)

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: distilgpt2
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
transformer.h.{0, 1, 2, 3, 4, 5}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


And some `TrainingArguments`:

In [81]:
from transformers import Trainer, TrainingArguments

model_name = model_checkpoint.split("/")[-1]
training_args = TrainingArguments(
    f"{model_name}-finetuned-wikitext2",
    eval_strategy = "epoch",
    logging_steps=10,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

The last argument is what enables **pushing your model to the Hugging Face Hub** automatically during training.

* If you **completed the setup steps at the top of the notebook** (installed the required tools and logged in to Hugging Face), you can keep this option enabled.
* If you **didn’t set up authentication**, simply remove this argument and the model will be trained and saved locally instead.

If you want more control over where and how the model is saved or pushed:

* To **save the model locally under a different name** than the Hub repository, or
* To **push the model to an organization** instead of your personal account,

you can use the `hub_model_id` argument. This should be the **full repository name**, including the namespace, for example:

* `"nuzaer/gpt-finetuned-wikitext2"`
* `"huggingface/gpt-finetuned-wikitext2"`

This gives you flexibility to organize your models exactly the way you want, whether locally or on the Hub.

We pass along all of those to the `Trainer` class:

In [82]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
)

In [83]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,4.143915
2,3.972348,4.129532
3,3.903941,4.125350


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=24, training_loss=3.930894136428833, metrics={'train_runtime': 288.0805, 'train_samples_per_second': 0.594, 'train_steps_per_second': 0.083, 'total_flos': 5585218043904.0, 'train_loss': 3.930894136428833, 'epoch': 3.0})

Once training is finished, we can **evaluate how well the model learned** by running it on the **validation set**.

From this evaluation, we compute **perplexity**, which is a standard metric for language models. In simple terms, perplexity tells us **how surprised the model is by the text**:

* **Lower perplexity** means the model is better at predicting the next token.
* **Higher perplexity** means the model is less confident and makes poorer predictions.

The following step shows how to run the evaluation and obtain the model’s perplexity on the validation data.


In [84]:
import math
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 61.89


Typical ranges you'll see:
| Perplexity | Interpretation                                    |
| ---------: | ------------------------------------------------- |
|    **1–5** | Extremely good (often unrealistic or overfitting) |
|   **5–20** | Very strong language model                        |
|  **20–50** | Reasonable / decent performance                   |
| **50–100** | Weak model                                        |
|   **100+** | Very poor / near-random predictions               |


Now that training is complete, you can **upload your trained model to the Hugging Face Hub**.

All you need to do is **run the next command**, and your model (along with its configuration and tokenizer) will be pushed to the Hub. Once uploaded, it will be available for you—and others—to use, share, and download.

Just make sure you’re logged in to your Hugging Face account before running it.


In [85]:
trainer.push_to_hub()

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...kitext2/training_args.bin: 100%|##########| 5.20kB / 5.20kB            

  ...kitext2/model.safetensors:   8%|7         | 25.1MB /  328MB            

CommitInfo(commit_url='https://huggingface.co/Nuzaer/distilgpt2-finetuned-wikitext2/commit/360bf67db5016776b307600426dc31bd27cbe47e', commit_message='End of training', commit_description='', oid='360bf67db5016776b307600426dc31bd27cbe47e', pr_url=None, repo_url=RepoUrl('https://huggingface.co/Nuzaer/distilgpt2-finetuned-wikitext2', endpoint='https://huggingface.co', repo_type='model', repo_id='Nuzaer/distilgpt2-finetuned-wikitext2'), pr_revision=None, pr_num=None)

You can now share this model with anyone: they can all load it with the identifier `"your-username/the-name-you-picked"` so for instance:

```python
# Load model directly
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("Nuzaer/distilgpt2-finetuned-wikitext2")
model = AutoModelForCausalLM.from_pretrained("Nuzaer/distilgpt2-finetuned-wikitext2")
```

## Masked language modeling

For **masked language modeling (MLM)**, we’ll prepare the dataset almost the same way as before, just with one extra step:

* We **randomly hide (mask) some tokens** in the input by replacing them with **`[MASK]`**.
* During training, the model’s job is to **guess the original masked tokens**.
* The **labels are set up so that only the masked positions count** toward the loss. In other words, the model is **not penalized** for the tokens that were not masked.

In this example, we’ll fine-tune **`distilroberta-base`**, which is a lightweight RoBERTa-style model trained for MLM.
If you’d like, you can also choose any other **masked language model checkpoint** from the list of available models on the Hub.


In [86]:
model_checkpoint = "distilroberta-base"

We can apply the same tokenization function as before, we just need to update our tokenizer to use the checkpoint we just picked:

In [87]:
tokenizer = AutoTokenizer.from_pretrained(model_checkpoint, use_fast=True)
tokenized_datasets = small_datasets.map(tokenize_function, batched=True, num_proc=4, remove_columns=["text"])

config.json:   0%|          | 0.00/480 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

Just like before, we **group multiple texts together and split them into fixed-length chunks** of size `block_size`. This helps the model see longer, continuous sequences during training, which is especially useful when working with long documents.

If your dataset already consists of **short, independent sentences** (for example, one sentence per example), you can **skip this step entirely** and feed those sentences directly to the model without chunking.


In [88]:
lm_datasets = tokenized_datasets.map(
    group_texts,
    batched=True,
    batch_size=1000,
    num_proc=4,
)

Map (num_proc=4):   0%|          | 0/100 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

Map (num_proc=4):   0%|          | 0/50 [00:00<?, ? examples/s]

Everything else stays mostly the same as before, with **two small but important differences**.

First, instead of a causal language model, we now use a model that is **designed specifically for masked language modeling (MLM)**. These models are trained to predict missing (masked) tokens using context from **both sides** of the sentence, rather than predicting the next token only.

So here, we load a **masked-LM–compatible model** (like `distilroberta-base`) to match the MLM training objective.


In [89]:
from transformers import AutoModelForMaskedLM
model = AutoModelForMaskedLM.from_pretrained(model_checkpoint)

model.safetensors:   0%|          | 0.00/331M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/106 [00:00<?, ?it/s]

RobertaForMaskedLM LOAD REPORT from: distilroberta-base
Key                         | Status     |  | 
----------------------------+------------+--+-
roberta.pooler.dense.bias   | UNEXPECTED |  | 
roberta.pooler.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


We redefine our `TrainingArguments`:

In [94]:
model_name = model_checkpoint.split("/")[-1]
training_args = TrainingArguments(
    f"{model_name}-finetuned-wikitext2",
    eval_strategy = "epoch",
    logging_steps=10,
    learning_rate=2e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    push_to_hub=True,
)

As before, the final argument is what allows the model to be **pushed to the Hugging Face Hub automatically during training**.

* If you **completed the setup steps at the beginning of the notebook** (installed the required tools and logged in), you can keep this option enabled.
* If you **didn’t do the setup**, simply remove this argument and the model will be trained and saved locally only.

If you want more control over where the model is saved or uploaded:

* To **save the model locally with a different name** than the Hub repository, or
* To **push the model under an organization account** instead of your personal account,

you can use the `hub_model_id` argument. This must be the **full repository name**, including the namespace, for example:

* `"Nuzaer/bert-finetuned-wikitext2"`
* `"huggingface/bert-finetuned-wikitext2"`

This lets you organize and share your models exactly the way you want.


Finally, we’ll use a special **`data_collator`**.

A `data_collator` is the function that takes individual examples from the dataset and **packages them into batches of tensors** that the model can train on.

* In the previous (CLM) example, batching was straightforward, so the default collator was enough.
* For **masked language modeling (MLM)**, we also need to **randomly mask tokens** (replace them with `[MASK]`) before feeding them to the model.

We *could* mask tokens ahead of time during preprocessing, but then the masking would be **exactly the same every epoch**, which isn’t ideal.
By doing masking inside the `data_collator`, we make sure that **each epoch masks different tokens**, giving the model more varied training examples.

Hugging Face provides a ready-made collator for this:
**`DataCollatorForLanguageModeling`**

And we can control how often tokens get masked by setting the **masking probability** (e.g., 15%).


In [95]:
from transformers import DataCollatorForLanguageModeling
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm_probability=0.15)

Then we just have to pass everything to `Trainer` and begin training:

In [96]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=lm_datasets["train"],
    eval_dataset=lm_datasets["validation"],
    data_collator=data_collator,
)

In [97]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,No log,2.300025
2,1.852345,2.332089
3,2.306695,2.189596


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:668: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  warnings.warn(warn_msg)


TrainOutput(global_step=24, training_loss=2.1688477198282876, metrics={'train_runtime': 276.4879, 'train_samples_per_second': 0.629, 'train_steps_per_second': 0.087, 'total_flos': 5769048586752.0, 'train_loss': 2.1688477198282876, 'epoch': 3.0})

Just like before, we can **evaluate the model on the validation set** once training is finished.

You’ll notice that the **perplexity is much lower** than what we saw with the causal language modeling (CLM) objective. This is expected and completely normal.

The reason is that **masked language modeling (MLM) is an easier task**:

* The model only needs to **predict the masked tokens**, which typically make up about **15% of the input**.
* At the same time, it can **see all the other tokens** in the sentence, both before and after the mask.

Because the model has more context and fewer tokens to predict, it performs better on this metric, resulting in a lower perplexity compared to CLM.


In [98]:
eval_results = trainer.evaluate()
print(f"Perplexity: {math.exp(eval_results['eval_loss']):.2f}")

Perplexity: 8.29


You can now upload the result of the training to the Hub, just execute this instruction:

In [99]:
# trainer.push_to_hub()